In [77]:
import os
import sys
import pandas as pd
import scanpy as sc

In [78]:
# ---------------------------------------------------------------------------------------------------
# This script computes pseudobulk aggregation (mean) for every gene at the cell type and sample level
# then combines results across cell types, such that there is only one file (per gene), with 28 
# observations representing the 28 cell types.
# This is to have an all-cells eQTL map to compare out cell type resolved results.
# ---------------------------------------------------------------------------------------------------

In [79]:
outdir = "/directflow/SCCGGroupShare/projects/anncuo/TenK10K_pilot/tenk10k/review_files/all_celltypes_pseudobulk"

In [80]:
anndata_dir = '/directflow/SCCGGroupShare/projects/blabow/tenk10k_phase1/data_processing/scanpy/output/integrated_objects/300_libraries/cpg_anndata_filtered/'

In [87]:
chrom = '10'

In [88]:
chrom_out_folder = f'{outdir}/chr{chrom}'
if not os.path.exists(chrom_out_folder):
    os.mkdir(chrom_out_folder)

In [89]:
celltypes='ASDC,B_intermediate,B_memory,B_naive,CD14_Mono,CD16_Mono,CD4_CTL,CD4_Naive,CD4_Proliferating,CD4_TCM,CD4_TEM,CD8_Naive,CD8_Proliferating,CD8_TCM,CD8_TEM,cDC1,cDC2,dnT,gdT,HSPC,ILC,MAIT,NK,NK_CD56bright,NK_Proliferating,pDC,Plasmablast,Treg'
celltypes = celltypes.split(',')
celltypes

['ASDC',
 'B_intermediate',
 'B_memory',
 'B_naive',
 'CD14_Mono',
 'CD16_Mono',
 'CD4_CTL',
 'CD4_Naive',
 'CD4_Proliferating',
 'CD4_TCM',
 'CD4_TEM',
 'CD8_Naive',
 'CD8_Proliferating',
 'CD8_TCM',
 'CD8_TEM',
 'cDC1',
 'cDC2',
 'dnT',
 'gdT',
 'HSPC',
 'ILC',
 'MAIT',
 'NK',
 'NK_CD56bright',
 'NK_Proliferating',
 'pDC',
 'Plasmablast',
 'Treg']

In [90]:
# get donor level covariates
sample_covs_file = f'{outdir}/saige-qtl_tenk10k-genome-2-3-eur_input_files_241210_covariates_sex_age_geno_pcs_shuffled_ids_tob_bioheart.csv'
sample_covs_df = pd.read_csv(sample_covs_file)
sample_covs_df['individual'] = sample_covs_df['sample_id']
sample_covs_df['cpg_id'] = sample_covs_df['sample_id']
sample_covs_df.index = sample_covs_df['cpg_id']
sample_covs_df.head()

,sample_id,sex,age,geno_PC1,geno_PC2,geno_PC3,geno_PC4,geno_PC5,geno_PC6,geno_PC7,...,sample_perm2,sample_perm3,sample_perm4,sample_perm5,sample_perm6,sample_perm7,sample_perm8,sample_perm9,individual,cpg_id
cpg_id,,,,,,,,,,,,,,,,,,,,,
CPG247296,CPG247296,1.0,25.0,0.040734,0.037115,0.034201,0.044705,0.012841,-0.003729,-0.080702,...,CPG314021,CPG308460,CPG249912,CPG310391,CPG500017,CPG313643,CPG309724,CPG250639,CPG247296,CPG247296
CPG247304,CPG247304,1.0,47.0,0.113983,0.082424,0.065252,0.064072,-0.066057,0.077341,0.021326,...,CPG305391,CPG249417,CPG309914,CPG501320,CPG313809,CPG498931,CPG310912,CPG501338,CPG247304,CPG247304
CPG247312,CPG247312,1.0,49.0,0.006067,0.015679,0.006319,0.009630,0.014275,-0.003539,-0.030022,...,CPG316133,CPG315572,CPG499269,CPG308536,CPG508143,CPG310490,CPG309500,CPG317230,CPG247312,CPG247312
CPG247320,CPG247320,1.0,51.0,0.004333,0.019983,0.007736,0.010266,0.009132,-0.003501,-0.033603,...,CPG305672,CPG317487,CPG247973,CPG247445,CPG252189,CPG251819,CPG250381,CPG500132,CPG247320,CPG247320
CPG247338,CPG247338,2.0,70.0,0.005765,0.020226,0.006217,0.017815,0.012736,-0.004748,-0.029675,...,CPG305565,CPG305508,CPG313015,CPG312512,CPG309724,CPG249490,CPG314716,CPG311449,CPG247338,CPG247338


In [91]:
for celltype in celltypes:
    
    #### Open AnnData object to extract single-cell expression
    # read in the data
    adata = sc.read(
        f'{anndata_dir}{celltype}_chr{chrom}.h5ad',
        cache=True,
    )
    donor_mapping = adata.obs["cpg_id"]
    # aggregate counts at the individual level (mean)
    pbcounts = adata.to_df().join(donor_mapping).groupby("cpg_id").agg("mean").sort_index()
    # prepare the pseudobulk metadata to create a new 'pseudobulk' AnnData object
    obs_pb = (
        adata.obs[["cpg_id", "cohort"]]
        .copy()
        .drop_duplicates()
        .set_index("cpg_id")
        .sort_index()
    )

    obs_pb["BioHEART"] = obs_pb["cohort"].map({"BioHEART": 1, "TOB": 0})
    
    # make the pseuodbulk AnnData object
    adata_pb = sc.AnnData(X=pbcounts, obs=obs_pb, var=adata.var)
    # re-calculate qc metrics (pseudobulk level)
    sc.pp.calculate_qc_metrics(
        adata_pb, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True
    )
    sc.pp.normalize_total(adata_pb, target_sum=1e6)
    sc.pp.log1p(adata_pb)
    sc.pp.highly_variable_genes(adata_pb, min_mean=0.0125, max_mean=3, min_disp=0.5)
    adata_pb= adata_pb[:, adata_pb.var.highly_variable]

    # calculate expression PCs
    sc.pp.scale(adata_pb, max_value=10)
    sc.tl.pca(adata_pb, svd_solver="arpack")
    donor_pca = pd.DataFrame(adata_pb.obsm["X_pca"])
    donor_pca.index = adata_pb.obs.index
    donor_pca.columns = [f"PC{i+1}" for i in donor_pca.columns]

    # pseudobulk level covariates
    pseudobulk_covariates = pd.DataFrame(
        adata_pb.obs[["BioHEART", "total_counts", "pct_counts_mt"]]
    )

    # combine covariate and pseudobulk expression for the relevant gene
    for gene in pbcounts.columns.values:
        out_file = f'{chrom_out_folder}/{gene}_{celltype}_pheno_cov.tsv'
        if not os.path.exists(out_file):
            pheno_cell_cov_df = pd.concat([pd.DataFrame(pbcounts[gene]), pseudobulk_covariates, donor_pca], axis=1)
            pheno_cell_cov_df['individual'] = pheno_cell_cov_df.index

            # combine all 
            pheno_cov_df = pheno_cell_cov_df.merge(sample_covs_df, on='individual', how='inner')
            pheno_cov_df['pseudocell'] = [f'{cpg_id}_{celltype}' for cpg_id in pheno_cov_df['cpg_id']]
            pheno_cov_df.to_csv(out_file)

/share/ScratchGeneral/anncuo/jupyter/conda_notebooks/envs/scanpy/lib/python3.6/site-packages/anndata/_core/anndata.py:120: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/share/ScratchGeneral/anncuo/jupyter/conda_notebooks/envs/scanpy/lib/python3.6/site-packages/scanpy/preprocessing/_simple.py:845: UserWarning: Revieved a view of an AnnData. Making a copy.
  view_to_actual(adata)
/share/ScratchGeneral/anncuo/jupyter/conda_notebooks/envs/scanpy/lib/python3.6/site-packages/anndata/_core/anndata.py:120: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/share/ScratchGeneral/anncuo/jupyter/conda_notebooks/envs/scanpy/lib/python3.6/site-packages/scanpy/preprocessing/_simple.py:845: UserWarning: Revieved a view of an AnnData. Making a copy.
  view_to_actual(adata)
/share/ScratchGeneral/anncuo/jupyter/conda_notebooks/envs/scanpy/l

/share/ScratchGeneral/anncuo/jupyter/conda_notebooks/envs/scanpy/lib/python3.6/site-packages/anndata/_core/anndata.py:120: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/share/ScratchGeneral/anncuo/jupyter/conda_notebooks/envs/scanpy/lib/python3.6/site-packages/scanpy/preprocessing/_simple.py:845: UserWarning: Revieved a view of an AnnData. Making a copy.
  view_to_actual(adata)
/share/ScratchGeneral/anncuo/jupyter/conda_notebooks/envs/scanpy/lib/python3.6/site-packages/anndata/_core/anndata.py:120: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/share/ScratchGeneral/anncuo/jupyter/conda_notebooks/envs/scanpy/lib/python3.6/site-packages/scanpy/preprocessing/_simple.py:845: UserWarning: Revieved a view of an AnnData. Making a copy.
  view_to_actual(adata)
/share/ScratchGeneral/anncuo/jupyter/conda_notebooks/envs/scanpy/l

In [92]:
chrom_out_folder

'/directflow/SCCGGroupShare/projects/anncuo/TenK10K_pilot/tenk10k/review_files/all_celltypes_pseudobulk/chr10'